In [30]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import torch
print(torch.cuda.is_available())

import keras
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# --- 1. Configuration ---
DATA_DIR = r"E:\KOULU\IRMAS\IRMAS-TrainingData" 
TEST_DIR = r"E:\KOULU\IRMAS\IRMAS-TestingData\Part1Part2Part3"
BATCH_SIZE = 32
IMG_SHAPE = (128, 130, 1)
PRED_THRESHOLD = 0.20

instrument_list = ['cel', 'cla', 'flu', 'gac', 'gel', 'org', 'pia', 'sax', 'tru', 'vio', 'voi']
label_map = {inst: i for i, inst in enumerate(instrument_list)}
NUM_CLASSES = len(instrument_list)

instrument_map = {
    'cel': 'Celesta',         'cla': 'Clarinet',        'flu': 'Flute',
    'gac': 'Acoustic Guitar', 'gel': 'Electric Guitar', 'org': 'Organ',
    'pia': 'Piano',           'sax': 'Saxophone',       'tru': 'Trumpet',
    'vio': 'Violin',          'voi': 'Human Voice'
}

True


In [31]:
# --- 2. File Discovery ---
file_paths = []
labels = []

for root, dirs, files in os.walk(DATA_DIR):
    folder_name = os.path.basename(root)
    if folder_name in label_map:
        idx = label_map[folder_name]
        for file in files:
            if file.endswith(".wav"):
                file_paths.append(os.path.join(root, file))
                # Change: labels are float32 vectors of shape (11,), each position represents one instrument (0.0 or 1.0)
                label_vec = np.zeros(NUM_CLASSES, dtype=np.float32)
                label_vec[idx] = 1.0
                labels.append(label_vec)

labels = np.array(labels, dtype=np.float32)  # Change: shape (N, 11) instead of (N,)
print(f"Found {len(file_paths)} training files")

# Split training data to train/val
# Change: stratify uses argmax because labels are now vectors
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42,
    stratify=np.argmax(labels, axis=1)
)
print(f"Train: {len(train_paths)} | Val: {len(val_paths)}")

# Compute weights (Added in v4)
# Convert one-hot to integer labels for weight computation
train_labels_int = np.argmax(train_labels, axis=1)
classes = np.unique(train_labels_int)
weights = compute_class_weight('balanced', classes=classes, y=train_labels_int)
class_weight_dict = dict(enumerate(weights))

print("Class weights:", class_weight_dict)

Found 6705 training files
Train: 5364 | Val: 1341
Class weights: {0: np.float64(1.5730205278592375), 1: np.float64(1.207020702070207), 2: np.float64(1.3507932510702594), 3: np.float64(0.9561497326203209), 4: np.float64(0.8020334928229665), 5: np.float64(0.8931068931068931), 6: np.float64(0.8451236804789665), 7: np.float64(0.9733260751224823), 8: np.float64(1.0577795306645632), 9: np.float64(1.0509404388714734), 10: np.float64(0.7839812920198772)}


In [32]:
# Converts raw .wav files into normalized Mel-spectrogram images for the CNN to "see"
def process_audio(file_path, label):
    def _parse_audio(fp):
        # Convert the TensorFlow tensor back into a standard Python string (file path)
        fp = fp.numpy().decode('utf-8')
        
        # Load audio file: 22050 samples per second, restricted to exactly 3.0 seconds
        audio, _ = librosa.load(fp, sr=22050, duration=3.0)
        
        # If audio is shorter than 3s (66,150 samples), add zeros to the end
        if len(audio) < 66150:
            audio = np.pad(audio, (0, 66150 - len(audio)))
        
        # Convert raw sound waves into a Mel-spectrogram (frequency over time)
        # n_mels=128 creates 128 vertical "frequency bins"
        spec = librosa.feature.melspectrogram(y=audio, sr=22050, n_mels=128)
        
        # Convert amplitude units to Decibels (log scale) so details aren't lost
        spec_db = librosa.power_to_db(spec, ref=np.max)
        
        # Normalize: shift -80..0 range to 0..1 for faster training (v2)
        spec_db = (spec_db + 80.0) / 80.0
        spec_db = np.clip(spec_db, 0, 1)
        
        # Add a gray channel dimension (128, 130) -> (128, 130, 1) to mimic a grayscale image
        return spec_db[..., np.newaxis].astype(np.float32)

    # Wrap the Python function so TensorFlow can run it as part of the "Graph"
    [image,] = tf.py_function(_parse_audio, [file_path], [tf.float32])
    image.set_shape(IMG_SHAPE)
    
    return image, label


In [33]:
# Builds an optimized, shuffled, and batched data pipeline to feed the GPU efficiently
def make_dataset(paths, labels):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.shuffle(len(paths))
    ds = ds.map(process_audio, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_paths, train_labels)
val_ds   = make_dataset(val_paths,   val_labels)

In [34]:
import torch.nn as nn

@keras.saving.register_keras_serializable()
class SimpleSpecAugment(nn.Module):
    """PyTorch-native SpecAugment for use with Keras+Torch backend."""
    
    def __init__(self, time_masks=3, freq_masks=3, time_mask_param=50, 
                 freq_mask_param=25, noise_std=0.01):
        super().__init__()
        self.time_masks = time_masks
        self.freq_masks = freq_masks
        self.time_mask_param = time_mask_param
        self.freq_mask_param = freq_mask_param
        self.noise_std = noise_std
        
    def forward(self, inputs, training=True):
        if not training:
            return inputs
            
        # inputs shape: (batch, height, width, channels)
        result = inputs.clone()
        batch_size, h, w, c = result.shape

        # Add Gaussian noise
        if self.noise_std > 0:
            noise = torch.randn_like(result) * self.noise_std
            result = result + noise
        
        # Frequency masking (along height dimension)
        for _ in range(self.freq_masks):
            # Random mask size and position for each batch item
            mask_size = torch.randint(0, self.freq_mask_param, (1,)).item()
            if mask_size >= h:
                continue
                
            mask_start = torch.randint(0, h - mask_size, (1,)).item()
            
            # Apply mask
            result[:, mask_start:mask_start + mask_size, :, :] = 0
        
        # Time masking (along width dimension)
        for _ in range(self.time_masks):
            mask_size = torch.randint(0, self.time_mask_param, (1,)).item()
            if mask_size >= w:
                continue
                
            mask_start = torch.randint(0, w - mask_size, (1,)).item()
            
            # Apply mask
            result[:, :, mask_start:mask_start + mask_size, :] = 0
            
        return result
    
    def get_config(self):
        """Returns the configuration of the layer for serialization."""
        return {
            'time_masks': self.time_masks,
            'freq_masks': self.freq_masks,
            'time_mask_param': self.time_mask_param,
            'freq_mask_param': self.freq_mask_param,
            'noise_std': self.noise_std,
        }

In [ ]:
@keras.saving.register_keras_serializable()
class SpecAugmentWrapper(tf.keras.layers.Layer):
    """Wrapper that applies SpecAugment using PyTorch operations."""
    # kwargs to pass through to base keras layer, exception otherwise
    def __init__(self, time_masks=2, freq_masks=2, time_mask_param=40, freq_mask_param=20, noise_std=0.01, **kwargs):
        super().__init__(**kwargs)
        self.time_masks = time_masks
        self.freq_masks = freq_masks
        self.time_mask_param = time_mask_param
        self.freq_mask_param = freq_mask_param
        self.noise_std = noise_std
        # Create the PyTorch module
        self.specaugment = SimpleSpecAugment(
            time_masks, freq_masks, time_mask_param, freq_mask_param, noise_std
        )
    
    def call(self, inputs, training=None):
        if training:
            # Apply SpecAugment
            return self.specaugment(inputs, training=True)
        return inputs
    
    def get_config(self):
        """Returns the configuration of the layer for serialization."""
        config = super().get_config()
        config.update({
            'time_masks': self.time_masks,
            'freq_masks': self.freq_masks,
            'time_mask_param': self.time_mask_param,
            'freq_mask_param': self.freq_mask_param,
            'noise_std': self.noise_std,
        })
        return config
    
    @classmethod
    def from_config(cls, config):
        """Creates the layer from its configuration."""
        return cls(**config)

In [36]:
model = tf.keras.models.load_model('irmas_instrument_classifier_v3_multilabel_data_aug.keras')

In [37]:
from sklearn.metrics import f1_score

# (Added in v4)
def find_optimal_thresholds(y_true, y_pred_probs, val_data):
    """Find best threshold per class using validation set"""
    optimal_thresholds = []
    
    for i in range(NUM_CLASSES):
        best_f1 = 0
        best_thresh = 0.2  # default
        
        for thresh in np.arange(0.1, 0.6, 0.05):
            y_pred_binary = (y_pred_probs[:, i] > thresh).astype(int)
            f1 = f1_score(y_true[:, i], y_pred_binary)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = thresh
                
        optimal_thresholds.append(best_thresh)
        print(f"{instrument_list[i]}: threshold={best_thresh:.3f}, F1={best_f1:.3f}")
    
    return optimal_thresholds

In [38]:
## THRESHOLD IS THE CONFIDENCE BAR WHICH DECIDES IF THERE IS AN X INSTRUMENT IN THE SAMPLE.
## WE HAVE 11 INSTRUMENTS SO RANDOM GUESSING WOULD BE AROUND 0.09.
## SO IF WE HAVE THRESHOLD OF 0.20 IT NEEDS TO HAVE ATLEAST 0.20 PROBABILITY TO BE FLAGGED
## THAT THE INSTRUMENT IS IN THE SAMPLE.

# Change: returns a probability vector (11,) instead of a single float for one instrument -> all instruments are evaluated in one pass per file
def predict_on_test_file(file_path, model):
    audio, sr = librosa.load(file_path, sr=22050)  # sample rate, 22,050 * 3 = 66,150 samples per chunk
    chunk_size = 66150  # 3 seconds
    chunk_probs = []

    for i in range(0, len(audio) - chunk_size, chunk_size):
        chunk = audio[i : i + chunk_size]
        spec    = librosa.feature.melspectrogram(y=chunk, sr=sr, n_mels=128)
        spec_db = librosa.power_to_db(spec, ref=np.max)  # v2
        spec_db = (spec_db + 80.0) / 80.0               # v2
        spec_db = np.clip(spec_db, 0, 1)                 # v2
        inp  = spec_db[np.newaxis, ..., np.newaxis]
        pred = model.predict(inp, verbose=0)
        chunk_probs.append(pred[0])  # CHANGED: shape (11,) instead of a single float

    # Return the highest confidence found across all 3s windows per instrument
    # If the instrument appears once, the whole file is flagged as a match (depending on the threshold)
    # Change: np.max over axis=0 to get per-instrument max across chunks
    return np.max(chunk_probs, axis=0) if chunk_probs else np.zeros(NUM_CLASSES)

def predict_with_dynamic_threshold(file_path, model, thresholds):
    probs = predict_on_test_file(file_path, model)
    return (probs > thresholds).astype(int)

test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.wav')])

# Change: y_true and y_pred_probs are now 2D arrays (num_files, 11) instead of 1D
y_true       = []
y_pred_probs = []
y_pred_binary = []

# Acquire optimal thresholds for instrument recog from validation set (Added in v4)
val_pred_probs = model.predict(val_ds)
optimal_thresholds = find_optimal_thresholds(val_labels, val_pred_probs, val_ds)

for f in test_files:
    file_path = os.path.join(TEST_DIR, f)
    txt_path  = os.path.splitext(file_path)[0] + '.txt'

    # Change: label is now a vector — all instruments mentioned in the .txt are flagged
    label_vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    if os.path.exists(txt_path):
        with open(txt_path, 'r') as t:
            labels_in_file = [item.strip() for item in t.read().split()]
            for code in labels_in_file:
                if code in label_map:
                    label_vec[label_map[code]] = 1.0

    y_true.append(label_vec)
    y_pred_binary.append(predict_with_dynamic_threshold(file_path, model, thresholds=optimal_thresholds))

    # raw probabilities for AUC score
    probs = predict_on_test_file(file_path, model)
    y_pred_probs.append(probs)

y_true        = np.array(y_true)
y_pred_binary = np.array(y_pred_binary)

print(f"Evaluated {len(test_files)} test files")

42/42 ━━━━━━━━━━━━━━━━━━━━ 6s 128ms/step
cel: threshold=0.100, F1=0.082
cla: threshold=0.100, F1=0.127
flu: threshold=0.100, F1=0.167
gac: threshold=0.100, F1=0.127
gel: threshold=0.100, F1=0.177
org: threshold=0.100, F1=0.106
pia: threshold=0.100, F1=0.095
sax: threshold=0.100, F1=0.141
tru: threshold=0.100, F1=0.115
vio: threshold=0.100, F1=0.127
voi: threshold=0.100, F1=0.154
Evaluated 2874 test files


In [39]:
# RECALL    = "Of all the X instruments in the test set, how many did the model actually find?"
# PRECISION = "When the model says it's a X instrument, how often is it actually a X instrument?"
# F1 SCORE  = harmonic mean of precision and recall "The balance of the two"

# Change: one report to cover all 11 instruments
print(f"Multilabel Classification Report (thresholds={optimal_thresholds}):\n")
print(classification_report(
    y_true,
    y_pred_binary,
    target_names=[instrument_map[k] for k in instrument_list]
))

Multilabel Classification Report (thresholds=[np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1), np.float64(0.1)]):

                 precision    recall  f1-score   support

        Celesta       0.12      0.71      0.20       111
       Clarinet       0.05      0.74      0.09        62
          Flute       0.09      0.84      0.16       163
Acoustic Guitar       0.29      0.64      0.40       535
Electric Guitar       0.51      0.83      0.63       942
          Organ       0.22      0.58      0.32       361
          Piano       0.57      0.45      0.50       995
      Saxophone       0.23      0.84      0.37       326
        Trumpet       0.10      0.64      0.17       167
         Violin       0.21      0.82      0.34       211
    Human Voice       0.59      0.87      0.70      1044

      micro avg       0.28      0.71      0.41      4917
      macro avg      

e:\KOULU\Neural-Networks-Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [40]:
from sklearn.metrics import roc_auc_score

auc_score = roc_auc_score(y_true, y_pred_probs, average='macro')
print(f"Test AUC: {auc_score:.4f}")

Test AUC: 0.7664
